In [2]:
import pandas as pd
import numpy as np
import re

Datset is manually extracted from: https://libraryguides.missouri.edu/c.php?g=1039894

In [ ]:
df = pd.read_csv('ATU_TMI.csv')
df.head()

,ATU Classification Type,AT Classification Type,Thompson Motif
0,NaN,NaN,NaN
1,ATU 1 The theft of fish,AT 1 Fox steals fish by playing dead,K371.1. Trickster throws fish off the wagon.
2,NaN,AT 1* Fox and rabbit steal a basket (now ATU 1),NaN
3,ATU 2 Tail-Fisher,AT 2 How the Bear Lost His Tail,K1021 The tail fisher.
4,ATU 2A Torn-off Tails,AT 2A Caught by the Tail,K1021.1. Tail buried (hair tied)


In [ ]:
df[10:20]

,ATU Classification Type,AT Classification Type,Thompson Motif
10,ATU 7 The three tree names,AT 7 Contest to Name Three Trees,N51 Wager: who can call three tree names first.
11,ATU 8 False Beauty Treatment,AT 8 Painting the Bear Red,"K1013.2 ""Painting"" on the haycock."
12,NaN,AT 8A Curly Hair from Boiling Water (now ATU 8),K1013. False beauty-doctor.
13,ATU 9 The unjust partner,AT 9 The unjust partner,K1251.1 Holding up the roof.
14,NaN,AT 9B Dividing the Harvest (now ATU 9),K171.2. Deceptive grain division: the corn and...
15,NaN,AT 9C In cooking dinner the fox's porridge is ...,K471. The substituted porridge.
16,ATU 10*** The Fall Over the Edge,AT 10*** Over the edge,K891.5.1. Animals (giants) enticed over precip...
17,ATU 15 Theft of Food by Playing Godfater,AT 15 Stealing the Partner's Butter,K372. Playing godfather.
18,ATU 20A Animals Caught In a Pit Eat One Anothe...,AT 20 Eat the Smallest One First (now ATU 20A),K1024. Beginning with the smallest
19,NaN,AT 20A The Animals in the Pit,K1024. Beginning with the smallest


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2425 entries, 0 to 2424
Data columns (total 3 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   ATU Classification Type  1055 non-null   object
 1   AT Classification Type   1062 non-null   object
 2   Thompson Motif           2349 non-null   object
dtypes: object(3)
memory usage: 57.0+ KB


Forward fill ATU Classification Types:

In [ ]:
df_clean = df.copy()
df_clean['ATU Classification Type'] = df_clean['ATU Classification Type'].fillna(method='ffill')
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2425 entries, 0 to 2424
Data columns (total 3 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   ATU Classification Type  2424 non-null   object
 1   AT Classification Type   1062 non-null   object
 2   Thompson Motif           2349 non-null   object
dtypes: object(3)
memory usage: 57.0+ KB


/tmp/ipython-input-2162126693.py:2: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_clean['ATU Classification Type'] = df_clean['ATU Classification Type'].fillna(method='ffill')


In [ ]:
df_clean[10:20]

,ATU Classification Type,AT Classification Type,Thompson Motif
10,ATU 7 The three tree names,AT 7 Contest to Name Three Trees,N51 Wager: who can call three tree names first.
11,ATU 8 False Beauty Treatment,AT 8 Painting the Bear Red,"K1013.2 ""Painting"" on the haycock."
12,ATU 8 False Beauty Treatment,AT 8A Curly Hair from Boiling Water (now ATU 8),K1013. False beauty-doctor.
13,ATU 9 The unjust partner,AT 9 The unjust partner,K1251.1 Holding up the roof.
14,ATU 9 The unjust partner,AT 9B Dividing the Harvest (now ATU 9),K171.2. Deceptive grain division: the corn and...
15,ATU 9 The unjust partner,AT 9C In cooking dinner the fox's porridge is ...,K471. The substituted porridge.
16,ATU 10*** The Fall Over the Edge,AT 10*** Over the edge,K891.5.1. Animals (giants) enticed over precip...
17,ATU 15 Theft of Food by Playing Godfater,AT 15 Stealing the Partner's Butter,K372. Playing godfather.
18,ATU 20A Animals Caught In a Pit Eat One Anothe...,AT 20 Eat the Smallest One First (now ATU 20A),K1024. Beginning with the smallest
19,ATU 20A Animals Caught In a Pit Eat One Anothe...,AT 20A The Animals in the Pit,K1024. Beginning with the smallest


extract atu_id from ATU Classification Type:

In [ ]:
def extract_id(text):
  if pd.isna(text):
    return None

  text = str(text).strip()

  # Pattern 1: "ATU [number][letters][asterisks]" or "AT [number][letters][asterisks]"
  match = re.search(r'(?:ATU|AT)\s+(\d+[A-Za-z]*\**)', text)
  if match:
    return match.group(1)

  # Pattern 2: "ATU na" (special case)
  if re.search(r'ATU\s+na\b', text, re.IGNORECASE):
    return 'na'

  # Pattern 3: Starts with number directly (no ATU/AT prefix)
  match = re.search(r'^(\d+[A-Za-z]*\**)', text)
  if match:
    return match.group(1)

  return None

df_clean['atu_id'] = df_clean['ATU Classification Type'].apply(extract_id)



In [ ]:
df_clean[10:20]

,ATU Classification Type,AT Classification Type,Thompson Motif,atu_id
10,ATU 7 The three tree names,AT 7 Contest to Name Three Trees,N51 Wager: who can call three tree names first.,7
11,ATU 8 False Beauty Treatment,AT 8 Painting the Bear Red,"K1013.2 ""Painting"" on the haycock.",8
12,ATU 8 False Beauty Treatment,AT 8A Curly Hair from Boiling Water (now ATU 8),K1013. False beauty-doctor.,8
13,ATU 9 The unjust partner,AT 9 The unjust partner,K1251.1 Holding up the roof.,9
14,ATU 9 The unjust partner,AT 9B Dividing the Harvest (now ATU 9),K171.2. Deceptive grain division: the corn and...,9
15,ATU 9 The unjust partner,AT 9C In cooking dinner the fox's porridge is ...,K471. The substituted porridge.,9
16,ATU 10*** The Fall Over the Edge,AT 10*** Over the edge,K891.5.1. Animals (giants) enticed over precip...,10***
17,ATU 15 Theft of Food by Playing Godfater,AT 15 Stealing the Partner's Butter,K372. Playing godfather.,15
18,ATU 20A Animals Caught In a Pit Eat One Anothe...,AT 20 Eat the Smallest One First (now ATU 20A),K1024. Beginning with the smallest,20A
19,ATU 20A Animals Caught In a Pit Eat One Anothe...,AT 20A The Animals in the Pit,K1024. Beginning with the smallest,20A


In [ ]:
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2425 entries, 0 to 2424
Data columns (total 4 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   ATU Classification Type  2424 non-null   object
 1   AT Classification Type   1062 non-null   object
 2   Thompson Motif           2349 non-null   object
 3   atu_id                   2424 non-null   object
dtypes: object(4)
memory usage: 75.9+ KB


In [ ]:
df_clean[df_clean['Thompson Motif'].isna()]

,ATU Classification Type,AT Classification Type,Thompson Motif,atu_id
0,NaN,NaN,NaN,None
2,ATU 1 The theft of fish,AT 1* Fox and rabbit steal a basket (now ATU 1),NaN,1
22,ATU 30 The Fox Tricks the Wolf into Falling in...,AT 30 The Fox Leads the Wolf into a Pit,NaN,30
40,ATU 47D Dog Wants to Imitate a Wolf,AT 47D The Dog Imitating a Wolf Wants to Slay ...,NaN,47D
91,ATU 103A* Cat Claims to be King and Receives F...,AT 103A* Cat as the King of Beasts,NaN,103A*
...,...,...,...,...
2379,ATU 1962 My Father's Baptism (Wedding),AT 1962 At My Father's Baptism,NaN,1962
2406,2030 The Old Woman and Her Pig,2030J Disobedient Child is Punished (now ATU 2...,NaN,2030
2418,2204 The Dog's Cigar,2204 False Expectations,NaN,2204
2419,2251 The Rabbit's Tail,2251 This Tale Would Have Been Longer,NaN,2251


Drop the rows whose value for Thomspon motif in nan

In [ ]:
df_clean = df_clean.dropna(subset=['Thompson Motif'])
df_clean = df_clean[df_clean['Thompson Motif'] != 'na']
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2337 entries, 1 to 2424
Data columns (total 4 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   ATU Classification Type  2337 non-null   object
 1   AT Classification Type   977 non-null    object
 2   Thompson Motif           2337 non-null   object
 3   atu_id                   2337 non-null   object
dtypes: object(4)
memory usage: 91.3+ KB


Drop AT Classification type column

In [ ]:
df_clean.drop(columns=["AT Classification Type"], inplace=True)
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2337 entries, 1 to 2424
Data columns (total 3 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   ATU Classification Type  2337 non-null   object
 1   Thompson Motif           2337 non-null   object
 2   atu_id                   2337 non-null   object
dtypes: object(3)
memory usage: 73.0+ KB


In [ ]:
df_clean.head()

,ATU Classification Type,Thompson Motif,atu_id
1,ATU 1 The theft of fish,K371.1. Trickster throws fish off the wagon.,1
3,ATU 2 Tail-Fisher,K1021 The tail fisher.,2
4,ATU 2A Torn-off Tails,K1021.1. Tail buried (hair tied),2A
5,ATU 2B Basket Tied to Wolf's Tail,K1021.2. Basket tied to wolf's tail and filled...,2B
6,ATU 3 Simulated Injury,K473. Sham blood and brains.,3


In [ ]:
df_clean['Thompson Motif']

,Thompson Motif
1,K371.1. Trickster throws fish off the wagon.
3,K1021 The tail fisher.
4,K1021.1. Tail buried (hair tied)
5,K1021.2. Basket tied to wolf's tail and filled...
6,K473. Sham blood and brains.
...,...
2417,Z13.2. Catch tale: teller is killed in his own...
2421,Miscellaneous Type
2422,Z11. Endless tales.
2423,Z17. Rounds.


Parse thompson motifs:

In [ ]:
def parse_motif(text):

  text = str(text).strip()
  if not text:
    return None, None

  text = re.sub(r'^†+', '', text)
  text = text.replace('\n', ' ').replace('\\', ' ')
  text = re.sub(r'\s+', ' ', text).strip()
  text = re.sub(r'\s*\.\.\.+$', '', text)

  match = re.match(r'^([A-Z]\d+(?:\.\d+)*)\.?\s*(.*)$', text)
  if match:
            motif_id = match.group(1)
            motif_text = match.group(2).strip()
            motif_text = re.sub(r'^†+', '', motif_text).strip()
            motif_text = re.sub(r'^' + re.escape(motif_id) + r'\.?\s*', '', motif_text).strip()
            motif_text = re.sub(r'\s*\.\.\.+$', '', motif_text)

            if motif_text.startswith('"') and motif_text.endswith('"'):
                motif_text = motif_text[1:-1].strip()

            return motif_id, motif_text if motif_text else None
  else:
            return None, text if text else None

df_clean[['thompson_motif_id', 'thompson_motif_text']] = df_clean['Thompson Motif'].apply(
        lambda x: pd.Series(parse_motif(x))
    )

In [ ]:
df_clean[10:20]

,ATU Classification Type,Thompson Motif,atu_id,thompson_motif_id,thompson_motif_text
12,ATU 8 False Beauty Treatment,K1013. False beauty-doctor.,8,K1013,False beauty-doctor.
13,ATU 9 The unjust partner,K1251.1 Holding up the roof.,9,K1251.1,Holding up the roof.
14,ATU 9 The unjust partner,K171.2. Deceptive grain division: the corn and...,9,K171.2,Deceptive grain division: the corn and the chaff.
15,ATU 9 The unjust partner,K471. The substituted porridge.,9,K471,The substituted porridge.
16,ATU 10*** The Fall Over the Edge,K891.5.1. Animals (giants) enticed over precip...,10***,K891.5.1,Animals (giants) enticed over precipice.
17,ATU 15 Theft of Food by Playing Godfater,K372. Playing godfather.,15,K372,Playing godfather.
18,ATU 20A Animals Caught In a Pit Eat One Anothe...,K1024. Beginning with the smallest,20A,K1024,Beginning with the smallest
19,ATU 20A Animals Caught In a Pit Eat One Anothe...,K1024. Beginning with the smallest,20A,K1024,Beginning with the smallest
20,ATU 20C The animals flee in fear of the end of...,Z43.3. Nut hits cock in head: he thinks world ...,20C,Z43.3,Nut hits cock in head: he thinks world is comi...
21,ATU 21 Eating his own Entrails,K1025. Eating his own entrails.,21,K1025,Eating his own entrails.


In [ ]:
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2337 entries, 1 to 2424
Data columns (total 5 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   ATU Classification Type  2337 non-null   object
 1   Thompson Motif           2337 non-null   object
 2   atu_id                   2337 non-null   object
 3   thompson_motif_id        2327 non-null   object
 4   thompson_motif_text      2336 non-null   object
dtypes: object(5)
memory usage: 109.5+ KB


In [ ]:
df_clean[df_clean['thompson_motif_id'].isna()]

,ATU Classification Type,Thompson Motif,atu_id,thompson_motif_id,thompson_motif_text
488,ATU 425B Son of the Witch,Cupid and Psyche,425B,None,Cupid and Psyche
494,ATU 430 Donkey,641.4. Marriage to person in ass form.,430,None,641.4. Marriage to person in ass form.
510,433B King Lindorm,Transformation: man to bird,433B,None,Transformation: man to bird
634,ATU 505 Grateful dead,503 Hunchback and the Elves,505,None,503 Hunchback and the Elves
1078,ATU 672 Serpent's Crown,ccccc,672,None,ccccc
1614,ATU 910A Father's Precepts Disregarded,"21.4. ""Do not marry a girl from abroad"":",910A,None,"21.4. ""Do not marry a girl from abroad"":"
1998,ATU 1365 The obstinate wife,miscellaneous tales,1365,None,miscellaneous tales
2378,ATU 1960 Great Animal or Object,Miscellaneous Type,1960,None,Miscellaneous Type
2416,106 Animal's Conversation,Miscellaneous Type,106,None,Miscellaneous Type
2421,2271 Mock Stories for Children,Miscellaneous Type,2271,None,Miscellaneous Type


I manually fix the above columns:
To fill missing value si used folkmasa.org

In [ ]:
df_clean = df_clean[df_clean['Thompson Motif'] != 'Miscellaneous Type']
df_clean = df_clean[df_clean['Thompson Motif'] != 'miscellaneous tales']
df_clean = df_clean[df_clean['Thompson Motif'] != 'ccccc']

In [ ]:
df_clean[df_clean['thompson_motif_id'].isna()]

,ATU Classification Type,Thompson Motif,atu_id,thompson_motif_id,thompson_motif_text
488,ATU 425B Son of the Witch,Cupid and Psyche,425B,None,Cupid and Psyche
494,ATU 430 Donkey,641.4. Marriage to person in ass form.,430,None,641.4. Marriage to person in ass form.
510,433B King Lindorm,Transformation: man to bird,433B,None,Transformation: man to bird
634,ATU 505 Grateful dead,503 Hunchback and the Elves,505,None,503 Hunchback and the Elves
1614,ATU 910A Father's Precepts Disregarded,"21.4. ""Do not marry a girl from abroad"":",910A,None,"21.4. ""Do not marry a girl from abroad"":"


In [ ]:
df_clean['ATU Classification Type'][494]

'ATU 430 Donkey'

In [ ]:
df_clean['thompson_motif_id'][494] = "B641.4"
df_clean['thompson_motif_text'][494] = "Marriage to person in ass form"

df_clean['thompson_motif_id'][510] = "D150"

df_clean['thompson_motif_id'][1614] = "J21.4"
df_clean['thompson_motif_text'][1614] = "Do not marry a girl from abroad"



/tmp/ipython-input-2585529263.py:1: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df_clean['thompson_motif_id'][494] = "B641.4"
/tmp/ipython-input-2585529263.py:2: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
Y

In [ ]:
df_clean[df_clean['thompson_motif_id'].isna()]

,ATU Classification Type,Thompson Motif,atu_id,thompson_motif_id,thompson_motif_text
488,ATU 425B Son of the Witch,Cupid and Psyche,425B,None,Cupid and Psyche
634,ATU 505 Grateful dead,503 Hunchback and the Elves,505,None,503 Hunchback and the Elves


In [ ]:
df_clean = df_clean[df_clean['Thompson Motif'] != 'Cupid and Psyche']
df_clean = df_clean[df_clean['Thompson Motif'] != '503 Hunchback and the Elves']
df_clean[df_clean['thompson_motif_id'].isna()]

,ATU Classification Type,Thompson Motif,atu_id,thompson_motif_id,thompson_motif_text


In [ ]:
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2330 entries, 1 to 2424
Data columns (total 5 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   ATU Classification Type  2330 non-null   object
 1   Thompson Motif           2330 non-null   object
 2   atu_id                   2330 non-null   object
 3   thompson_motif_id        2330 non-null   object
 4   thompson_motif_text      2329 non-null   object
dtypes: object(5)
memory usage: 109.2+ KB


In [ ]:
df_clean.head()

,ATU Classification Type,Thompson Motif,atu_id,thompson_motif_id,thompson_motif_text
1,ATU 1 The theft of fish,K371.1. Trickster throws fish off the wagon.,1,K371.1,Trickster throws fish off the wagon.
3,ATU 2 Tail-Fisher,K1021 The tail fisher.,2,K1021,The tail fisher.
4,ATU 2A Torn-off Tails,K1021.1. Tail buried (hair tied),2A,K1021.1,Tail buried (hair tied)
5,ATU 2B Basket Tied to Wolf's Tail,K1021.2. Basket tied to wolf's tail and filled...,2B,K1021.2,Basket tied to wolf's tail and filled with sto...
6,ATU 3 Simulated Injury,K473. Sham blood and brains.,3,K473,Sham blood and brains.


In [ ]:
df_clean = df_clean.drop_duplicates()
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2241 entries, 1 to 2424
Data columns (total 5 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   ATU Classification Type  2241 non-null   object
 1   Thompson Motif           2241 non-null   object
 2   atu_id                   2241 non-null   object
 3   thompson_motif_id        2241 non-null   object
 4   thompson_motif_text      2240 non-null   object
dtypes: object(5)
memory usage: 105.0+ KB


In [ ]:
df_clean[df_clean['thompson_motif_text'].isna()]

,ATU Classification Type,Thompson Motif,atu_id,thompson_motif_id,thompson_motif_text
2137,ATU 1537 The corpse killed five times,K2151. †K2151.,1537,K2151,None


In [ ]:
df_clean['thompson_motif_text'][2137] = "The Corpse Handed Around"

/tmp/ipython-input-2814097329.py:1: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df_clean['thompson_motif_text'][2137] = "The Corpse Handed Around"


In [ ]:
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2241 entries, 1 to 2424
Data columns (total 5 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   ATU Classification Type  2241 non-null   object
 1   Thompson Motif           2241 non-null   object
 2   atu_id                   2241 non-null   object
 3   thompson_motif_id        2241 non-null   object
 4   thompson_motif_text      2241 non-null   object
dtypes: object(5)
memory usage: 169.6+ KB


In [ ]:
df_clean.to_csv("ATU_to_TMI.csv")

In [3]:
df = pd.read_csv("ATU_to_TMI.csv")
df.head()

,Unnamed: 0,ATU Classification Type,Thompson Motif,atu_id,thompson_motif_id,thompson_motif_text
0,1,ATU 1 The theft of fish,K371.1. Trickster throws fish off the wagon.,1,K371.1,Trickster throws fish off the wagon.
1,3,ATU 2 Tail-Fisher,K1021 The tail fisher.,2,K1021,The tail fisher.
2,4,ATU 2A Torn-off Tails,K1021.1. Tail buried (hair tied),2A,K1021.1,Tail buried (hair tied)
3,5,ATU 2B Basket Tied to Wolf's Tail,K1021.2. Basket tied to wolf's tail and filled...,2B,K1021.2,Basket tied to wolf's tail and filled with sto...
4,6,ATU 3 Simulated Injury,K473. Sham blood and brains.,3,K473,Sham blood and brains.


In [4]:
thompson_motif_df = pd.read_csv("https://github.com/KatjaMellmann/TMI_as_CSV/blob/main/tmi.csv?raw=true")
thompson_motif_df.fillna("", inplace=True)

thompson_motif_df.head()

,code,[sorting field],1st ed.,chapter,division1,division2,division3,"section (""tens"")",MOTIF,bibliographies
0,A0,A0000,A0,A. Mythological motifs.,A0–A99. Creator.,,,A0. Creator.,A0. Creator.,"For a general bibliography of creation myths, ..."
1,A1,A0001,,A. Mythological motifs.,A0–A99. Creator.,,,A0. Creator.,A1. Identity of creator.,
2,A1.1,A0001.1,A1,A. Mythological motifs.,A0–A99. Creator.,,,A0. Creator.,A1.1. Sun-god as creator.,Egyptian: Müller 69; Persian: Carnoy 260.
3,A1.2,A0001.2,,A. Mythological motifs.,A0–A99. Creator.,,,A0. Creator.,A1.2. Grandfather as creator.,S. Am. Indian (Paressi): Métraux BBAE CXLIII (...
4,A1.3,A0001.3,,A. Mythological motifs.,A0–A99. Creator.,,,A0. Creator.,A1.3. Stone-woman as creator.,Paressi: Métraux BBAE CXLIII (3) 359.


In [8]:
thompson_motif_df[thompson_motif_df['code']=="K371.1"]

,code,[sorting field],1st ed.,chapter,division1,division2,division3,"section (""tens"")",MOTIF,bibliographies
33322,K371.1,K0371.1,K371.1,K. Deceptions.,K300–K499. Thefts and cheats.,K310–K439. Thefts.,,K360. Other means of theft.,K371.1. Trickster throws fish off the wagon. T...,"*Type 1; BP II 116; Dh IV 225, 304; Krohn Bär ..."


In [7]:
thompson_motif_df[thompson_motif_df['code']=="K371"]

,code,[sorting field],1st ed.,chapter,division1,division2,division3,"section (""tens"")",MOTIF,bibliographies
33321,K371,K0371,K371,K. Deceptions.,K300–K499. Thefts and cheats.,K310–K439. Thefts.,,K360. Other means of theft.,K371. Trickster hides in food and eats it.,India: Thompson-Balys; Indonesia: DeVries's li...


In [9]:
thompson_motif_df.drop(columns=["[sorting field]", "1st ed.", "bibliographies"], inplace=True)
thompson_motif_df.head()

,code,chapter,division1,division2,division3,"section (""tens"")",MOTIF
0,A0,A. Mythological motifs.,A0–A99. Creator.,,,A0. Creator.,A0. Creator.
1,A1,A. Mythological motifs.,A0–A99. Creator.,,,A0. Creator.,A1. Identity of creator.
2,A1.1,A. Mythological motifs.,A0–A99. Creator.,,,A0. Creator.,A1.1. Sun-god as creator.
3,A1.2,A. Mythological motifs.,A0–A99. Creator.,,,A0. Creator.,A1.2. Grandfather as creator.
4,A1.3,A. Mythological motifs.,A0–A99. Creator.,,,A0. Creator.,A1.3. Stone-woman as creator.


In [10]:
thompson_motif_df = thompson_motif_df.rename(columns={'section ("tens")': 'section_tens'})

def clean_text(text):
    if pd.isna(text) or not isinstance(text, str):
        return ""

    cleaned = re.sub(r'^[A-Z0-9.–]+\.?\s*', '', text.strip())
    cleaned = cleaned.rstrip('.').strip()
    return cleaned


thompson_motif_df['chapter_text']   = thompson_motif_df['chapter'].apply(clean_text)
thompson_motif_df['division1_text'] = thompson_motif_df['division1'].apply(clean_text)
thompson_motif_df['division2_text'] = thompson_motif_df['division2'].apply(clean_text)
thompson_motif_df['division3_text'] = thompson_motif_df['division3'].apply(clean_text)
thompson_motif_df['section_text']   = thompson_motif_df['section_tens'].apply(clean_text)
thompson_motif_df.head()

,code,chapter,division1,division2,division3,section_tens,MOTIF,chapter_text,division1_text,division2_text,division3_text,section_text
0,A0,A. Mythological motifs.,A0–A99. Creator.,,,A0. Creator.,A0. Creator.,Mythological motifs,Creator,,,Creator
1,A1,A. Mythological motifs.,A0–A99. Creator.,,,A0. Creator.,A1. Identity of creator.,Mythological motifs,Creator,,,Creator
2,A1.1,A. Mythological motifs.,A0–A99. Creator.,,,A0. Creator.,A1.1. Sun-god as creator.,Mythological motifs,Creator,,,Creator
3,A1.2,A. Mythological motifs.,A0–A99. Creator.,,,A0. Creator.,A1.2. Grandfather as creator.,Mythological motifs,Creator,,,Creator
4,A1.3,A. Mythological motifs.,A0–A99. Creator.,,,A0. Creator.,A1.3. Stone-woman as creator.,Mythological motifs,Creator,,,Creator


In [11]:
thompson_motif_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 46302 entries, 0 to 46301
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   code            46302 non-null  object
 1   chapter         46302 non-null  object
 2   division1       46302 non-null  object
 3   division2       46302 non-null  object
 4   division3       46302 non-null  object
 5   section_tens    46302 non-null  object
 6   MOTIF           46302 non-null  object
 7   chapter_text    46302 non-null  object
 8   division1_text  46302 non-null  object
 9   division2_text  46302 non-null  object
 10  division3_text  46302 non-null  object
 11  section_text    46302 non-null  object
dtypes: object(12)
memory usage: 4.2+ MB


In [12]:

thompson_motif_df = thompson_motif_df.rename(columns={'code': 'thompson_motif_id'})

In [13]:
merged_df = df.merge(
    thompson_motif_df,
    on='thompson_motif_id',
    how='inner'
)

In [14]:
merged_df.head()

,Unnamed: 0,ATU Classification Type,Thompson Motif,atu_id,thompson_motif_id,thompson_motif_text,chapter,division1,division2,division3,section_tens,MOTIF,chapter_text,division1_text,division2_text,division3_text,section_text
0,1,ATU 1 The theft of fish,K371.1. Trickster throws fish off the wagon.,1,K371.1,Trickster throws fish off the wagon.,K. Deceptions.,K300–K499. Thefts and cheats.,K310–K439. Thefts.,,K360. Other means of theft.,K371.1. Trickster throws fish off the wagon. T...,Deceptions,Thefts and cheats,Thefts,,Other means of theft
1,3,ATU 2 Tail-Fisher,K1021 The tail fisher.,2,K1021,The tail fisher.,K. Deceptions.,K1000–K1199. Deception into self-injury.,,,K1020. Deception into disastrous attempt to pr...,K1021. The tail fisher. The bear is persuaded ...,Deceptions,Deception into self-injury,,,Deception into disastrous attempt to procure food
2,4,ATU 2A Torn-off Tails,K1021.1. Tail buried (hair tied),2A,K1021.1,Tail buried (hair tied),K. Deceptions.,K1000–K1199. Deception into self-injury.,,,K1020. Deception into disastrous attempt to pr...,K1021.1. Tail buried (hair tied). Dupe bound f...,Deceptions,Deception into self-injury,,,Deception into disastrous attempt to procure food
3,5,ATU 2B Basket Tied to Wolf's Tail,K1021.2. Basket tied to wolf's tail and filled...,2B,K1021.2,Basket tied to wolf's tail and filled with sto...,K. Deceptions.,K1000–K1199. Deception into self-injury.,,,K1020. Deception into disastrous attempt to pr...,K1021.2. Basket tied to wolf's tail and filled...,Deceptions,Deception into self-injury,,,Deception into disastrous attempt to procure food
4,6,ATU 3 Simulated Injury,K473. Sham blood and brains.,3,K473,Sham blood and brains.,K. Deceptions.,K300–K499. Thefts and cheats.,K440–K499. Other cheats.,,K440. Other cheats.,K473. Sham blood and brains. Fox covers his he...,Deceptions,Thefts and cheats,Other cheats,,Other cheats


In [15]:
merged_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2161 entries, 0 to 2160
Data columns (total 17 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   Unnamed: 0               2161 non-null   int64 
 1   ATU Classification Type  2161 non-null   object
 2   Thompson Motif           2161 non-null   object
 3   atu_id                   2161 non-null   object
 4   thompson_motif_id        2161 non-null   object
 5   thompson_motif_text      2161 non-null   object
 6   chapter                  2161 non-null   object
 7   division1                2161 non-null   object
 8   division2                2161 non-null   object
 9   division3                2161 non-null   object
 10  section_tens             2161 non-null   object
 11  MOTIF                    2161 non-null   object
 12  chapter_text             2161 non-null   object
 13  division1_text           2161 non-null   object
 14  division2_text           2161 non-null  

In [16]:
merged_df.to_csv("ATU_to_TMI.csv")